In [1]:
# 04_robustness.ipynb
# Multi-seed robustness check. Repeat the full data-generating process and analysis
# over 300 independent seeds and confirm the key findings are stable: the convergence
# index separates healthy pluralism from oversight failure (large Cohen's d), while
# first-stage dispersion does not (d near zero); the combined classifier beats
# dispersion alone (AUC); and the determinant signs recover with p<.05. Figures are
# greyscale; no file paths are printed.
import numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf, statsmodels.api as sm
from sklearn.metrics import roc_auc_score
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "font.size": 11,
                     "axes.edgecolor": "black", "axes.linewidth": 0.8})
FIG = Path("..")/"results"/"figures"; TAB = Path("..")/"results"/"tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def save(fig, name):
    fig.savefig(FIG/f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG/f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

N_BOARDS, N_SCEN = 140, 12
RELS = ["monotonicity", "equivalence", "ordering", "composition"]

# ----- One realization of the data-generating process for a given seed -----
def gen(seed):
    rng = np.random.default_rng(seed)
    z = lambda n: rng.normal(0, 1, n)
    ai = z(N_BOARDS); div = np.clip(rng.beta(2, 2, N_BOARDS), 0.02, 0.98)
    ind = np.clip(rng.beta(2.5, 2, N_BOARDS), 0.02, 0.98)
    ten = np.clip(rng.gamma(2.0, 3.0, N_BOARDS), 0.5, 25.0); rc = z(N_BOARDS)
    esc = rng.binomial(1, 0.5, N_BOARDS); tz = (ten - ten.mean())/ten.std()
    D = 1/(1+np.exp(-(2*0.9*(div-0.5) - 0.5*ai + 0.35*z(N_BOARDS))))
    C = 1/(1+np.exp(-(1.1*esc + 2*(ind-0.5) + 2*0.7*(div-0.5) + 0.6*tz + 0.3*rc + 0.35*z(N_BOARDS))))
    hd, hc = D >= np.median(D), C >= np.median(C)
    cls = np.where(~hd, "A", np.where(hc, "B", "C")); fail = (cls == "C").astype(int)
    rpb = rng.integers(1, 8, N_BOARDS); bo = np.repeat(np.arange(N_BOARDS), rpb); Nn = bo.size
    rai = ai[bo] + 0.5*z(Nn); rbias = 0.36*(0.5*rc[bo] + 0.8*z(Nn))
    sev = rng.uniform(2, 6, N_SCEN); rel = np.array([RELS[i % 4] for i in range(N_SCEN)])
    dl = rng.uniform(0.8, 1.6, N_SCEN); fol = sev.copy()
    for i, r in enumerate(rel):
        fol[i] = sev[i]-dl[i] if r == "monotonicity" else (sev[i]+dl[i] if r == "ordering"
                 else (sev[i]+0.5*dl[i] if r == "composition" else sev[i]))
    fol = np.clip(fol, 1, 7)
    def noisy(v):
        sd = np.clip(0.35 + 1.9*D[bo] - 0.35*rai, 0.2, 3.0)
        return np.clip(np.rint(v + rbias + rng.normal(0, sd)), 1, 7)
    def anch(v): return pd.Series(v).groupby(bo).transform("mean").to_numpy()
    w = C[bo]; fr = []
    for s in range(N_SCEN):
        s1, f1 = noisy(sev[s]), noisy(fol[s])
        s2 = np.clip(np.rint((1-w)*s1 + w*anch(s1) + rng.normal(0, 0.25, Nn)), 1, 7)
        f2 = np.clip(np.rint((1-w)*f1 + w*anch(f1) + rng.normal(0, 0.25, Nn)), 1, 7)
        for st, (a, b) in [(1, (s1, f1)), (2, (s2, f2))]:
            fr.append(pd.DataFrame({"bo": bo, "pair": s, "rel": rel[s], "stage": st, "src": a, "fol": b}))
    L = pd.concat(fr, ignore_index=True)
    rl = L.rel.to_numpy(); sc = L.src.to_numpy(); fo = L.fol.to_numpy(); v = np.zeros(len(L), int)
    v[(rl == "monotonicity") & (fo > sc)] = 1; v[(rl == "equivalence") & (fo != sc)] = 1
    v[(rl == "ordering") & (fo < sc)] = 1;     v[(rl == "composition") & (fo < sc)] = 1
    L["v"] = v
    disp = lambda d: d.groupby("pair")["src"].std(ddof=0).mean()
    rec = []
    for b, g in L.groupby("bo"):
        g1, g2 = g[g.stage == 1], g[g.stage == 2]; v1, v2 = g1.v.mean(), g2.v.mean()
        d1, d2 = disp(g1), disp(g2); d1 = 0 if np.isnan(d1) else d1; d2 = 0 if np.isnan(d2) else d2
        rec.append(dict(bo=b, viol1=v1, disp1=d1, conv_disp=(d1-d2)/d1 if d1 > 0 else 0,
                        n=int((g.stage == 1).sum()//N_SCEN)))
    B = pd.DataFrame(dict(bo=np.arange(N_BOARDS), ai=ai, div=div, ind=ind, tz=tz, rc=rc,
                          esc=esc, cls=cls, fail=fail)).merge(pd.DataFrame(rec), on="bo")
    return B[B.n >= 3].copy()

def cohend(x, y):
    nx, ny = len(x), len(y)
    sp = np.sqrt(((nx-1)*x.var(ddof=1) + (ny-1)*y.var(ddof=1))/(nx+ny-2))
    return (x.mean()-y.mean())/sp if sp > 0 else np.nan

# ----- Repeat across 300 independent seeds -----
rows = []
for seed in range(1000, 1300):
    b = gen(seed)
    for c in ["ai", "div", "ind", "tz", "rc"]:
        b[c+"_s"] = (b[c] - b[c].mean())/b[c].std()
    B, C = b[b.cls == "B"], b[b.cls == "C"]
    m1 = smf.ols("viol1 ~ ai_s+div_s+ind_s+esc+tz_s+rc_s", b).fit()
    m2 = smf.ols("disp1 ~ div_s+ai_s", b).fit()
    m3 = smf.ols("conv_disp ~ esc+ind_s+div_s+tz_s+rc_s", b).fit()
    Xc = sm.add_constant(b[["disp1", "conv_disp"]]); comb = sm.Logit(b.fail.values, Xc).fit(disp=0)
    rows.append(dict(
        d_disp=cohend(B.disp1, C.disp1), d_conv=cohend(B.conv_disp, C.conv_disp),
        auc_disp=roc_auc_score(b.fail, b.disp1), auc_comb=roc_auc_score(b.fail, comb.predict(Xc)),
        H1=int((m1.params["ai_s"] < 0) and (m1.pvalues["ai_s"] < 0.05)),
        H2=int((m2.params["div_s"] > 0) and (m2.pvalues["div_s"] < 0.05)),
        H3=int((m3.params["esc"] > 0) and (m3.pvalues["esc"] < 0.05))))
R = pd.DataFrame(rows)

# ----- Summary table (mean, 5th/95th percentiles) -----
def row(col): s = R[col]; return [round(s.mean(), 3), round(s.quantile(.05), 3), round(s.quantile(.95), 3)]
summary = pd.DataFrame(
    [["Cohen d: Stage-1 dispersion (B vs C)"] + row("d_disp"),
     ["Cohen d: convergence index (B vs C)"] + row("d_conv"),
     ["AUC: dispersion only"] + row("auc_disp"),
     ["AUC: dispersion + convergence"] + row("auc_comb"),
     ["H1 recovered (share of seeds)", round(R.H1.mean(), 3), "", ""],
     ["H2 recovered (share of seeds)", round(R.H2.mean(), 3), "", ""],
     ["H3 recovered (share of seeds)", round(R.H3.mean(), 3), "", ""]],
    columns=["metric", "mean", "p5", "p95"])
summary.to_csv(TAB/"table06_robustness_summary.csv", index=False)

# ----- Figure 6: Cohen's d across seeds (dispersion near 0, convergence large) -----
fig, ax = plt.subplots(figsize=(6.0, 3.8))
ax.hist(R.d_disp, bins=30, color="#bbbbbb", edgecolor="black", linewidth=0.5, label="Stage-1 dispersion")
ax.hist(R.d_conv, bins=30, color="#555555", edgecolor="black", linewidth=0.5, label="Convergence index")
ax.axvline(0, color="black", lw=0.9, ls="--")
ax.set_xlabel("Cohen's d (healthy pluralism vs failure)"); ax.set_ylabel("Number of seeds")
ax.legend(frameon=True, edgecolor="black")
save(fig, "fig06_robustness_cohend")

# ----- Figure 7: AUC across seeds -----
fig, ax = plt.subplots(figsize=(6.0, 3.8))
ax.hist(R.auc_disp, bins=25, color="#bbbbbb", edgecolor="black", linewidth=0.5, label="Dispersion only")
ax.hist(R.auc_comb, bins=25, color="#555555", edgecolor="black", linewidth=0.5, label="Dispersion + convergence")
ax.set_xlabel("AUC for classifying oversight failure"); ax.set_ylabel("Number of seeds")
ax.legend(frameon=True, edgecolor="black")
save(fig, "fig07_robustness_auc")

print("seeds:", len(R))
print(summary.to_string(index=False))
print("figures: fig06, fig07  | table: table06")


seeds: 300
                              metric   mean     p5    p95
Cohen d: Stage-1 dispersion (B vs C) -0.037 -0.538  0.442
 Cohen d: convergence index (B vs C)  2.587  2.024  3.172
                AUC: dispersion only  0.765  0.682   0.84
       AUC: dispersion + convergence  0.864  0.787  0.927
       H1 recovered (share of seeds)  1.000              
       H2 recovered (share of seeds)  1.000              
       H3 recovered (share of seeds)  1.000              
figures: fig06, fig07  | table: table06
